# Archived 2024 team experiment

Original source cells, with outputs and metadata removed for publication. This is a historical reference, not the supported runnable pipeline. It contains the evaluation limitations described in `docs/evaluation.md`, Colab dependencies, and original local path assumptions. Use the package and walkthrough for new runs.


If you want the code to run, First download the dataset at https://www.kaggle.com/datasets/andradaolteanu/gtzan-dataset-music-genre-classification

Then upload the dataset into your google drive.
once it is in the google drive, run code to mount the drive

Then put the filepath of the dataset into the below filepath variables.


# Data Processing


**Dataset Information from Kaggle** \
The GTZAN dataset, widely employed in machine listening research for music genre recognition (MGR), was compiled between 2000 and 2001. It comprises files sourced from diverse channels such as personal CDs, radio broadcasts, and microphone recordings. This diverse selection aims to encompass a wide range of recording conditions for comprehensive evaluation in MGR studies. Specifically, the GTZAN contains the 4 following datasets.

1. genres original - A collection of 10 genres with 100 audio files each, all having a length of 30 seconds.

2. images original - A visual representation for each audio file. One way to classify data is through neural networks. Because NNs (like CNN, what we will be using today) usually take in some sort of image representation, the audio files were converted to Mel Spectrograms to make this possible.

3. features_30_seconds.csv contains features of the audio files. It has for each song (30 seconds long) a mean and variance computed over multiple features (i.e. tempo, MFCC, RMS, etc.).

4. features_3_seconds.csv has the same structure, but the songs were split before into 3 seconds audio files (this way increasing 10 times the amount of data we fuel into our models).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import torch
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import scipy as sp
import PIL
from PIL import Image
import librosa as l
import os

## Data Processing for Primary Model (CNN): Cropping the Spectrograms
- getting rid of the redundant white space in the spectrograms

In [ ]:
folder_path = '/content/drive/MyDrive/APS360 Project/Data/images_original'

In [ ]:
# Create a list of image file paths in the folder
image_files = [os.path.join(folder_path, f) for f in os.listdir(folder_path) if f.endswith('.jpg')]

# Open each image file and resize it to 224x224
cropped_imgs = []
for image_file in image_files:
    with Image.open(image_file) as img:
        # The parameter values were obtained through trial and error
        left = 53.5
        upper = 34.6
        right = img.width - 41.5
        lower = img.height - 34.6
        CROP_DIMENSIONS = (left, upper, right, lower)

        # Crop the image to the bounding box
        cropped_img = img.crop(CROP_DIMENSIONS)
        cropped_imgs.append(cropped_img)

# Save the cropped images to a new folder
resized_folder_path = '/content/drive/MyDrive/APS360 Project/Data/images_cropped'
os.makedirs(resized_folder_path, exist_ok=True)
for i, cropped_img in enumerate(cropped_imgs):
    cropped_img.save(os.path.join(resized_folder_path, f'cropped_image_{i}.png'))

## Data Processing for Primary Model (CNN): Slice the Spectrograms

In [ ]:
from torchvision import datasets, transforms
from PIL import Image
from torch.utils.data import Dataset, DataLoader

class ChoppedDataset(Dataset):
    def __init__(self, folder_path, transform=None):
        self.folder_path = folder_path
        self.transform = transform
        self.slices = []
        self.labels = []

        # Load and transform all images
        for label, class_folder in enumerate(os.listdir(folder_path)):
            class_path = os.path.join(folder_path, class_folder)
            for image_name in os.listdir(class_path):
                image_path = os.path.join(class_path, image_name)
                image = Image.open(image_path)
                for i in range(32):  # chop each image into 32 slices
                    slice = image.crop((i * image.width // 32, 0, (i + 1) * image.width // 32, image.height))
                    if self.transform:
                        slice = self.transform(slice)
                    self.slices.append(slice)
                    self.labels.append(label)

    def __len__(self):
        return len(self.slices)

    def __getitem__(self, idx):
        return self.slices[idx], self.labels[idx]

def get_loaders(folder_path, batch_size):
    transform = transforms.Compose([
        transforms.Lambda(lambda x: x.convert('RGB')),  # Convert image to RGB
        transforms.Resize((64, 64)),  # Resize each slice to 64x64 - smaller than before as we have more samples to feed into CNN
        transforms.ToTensor()
    ])

    # Initialize the custom dataset
    dataset = ChoppedDataset(folder_path=folder_path, transform=transform)

    # Split the dataset into training, validation, and testing sets
    train_size = int(0.6 * len(dataset))
    val_size = int(0.2 * len(dataset))
    test_size = len(dataset) - (train_size + val_size)
    train_dataset, val_dataset, test_dataset = torch.utils.data.random_split(dataset, [train_size, val_size, test_size])

    # Create data loaders for each set
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

    return train_loader, val_loader, test_loader

# Specify the folder path and batch size
folder_path = '/content/drive/My Drive/APS360 Project/Data/images_original'
batch_size = 16

# Use the function to get data loaders
train_loader, val_loader, test_loader = get_loaders(folder_path, batch_size)

In [ ]:
# Get one batch from the training data loader
train_features, train_labels = next(iter(train_loader))

# Print the dimensions of the features and labels
print(f"Feature batch shape: {train_features.size()}")
print(f"Labels batch shape: {train_labels.size()}")


In [ ]:
num_data_points = 0
for _, _ in train_loader:
    num_data_points += _.size(0)  # Add the batch size of each iteration
for _, _ in val_loader:
    num_data_points += _.size(0)  # Add the batch size of each iteration
for _, _ in test_loader:
    num_data_points += _.size(0)  # Add the batch size of each iteration

print(f"Total number of data points in the training loader: {num_data_points}")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import torch

def show_image(tensor, one_channel=False):
    """
    Display an image from a tensor.

    Parameters:
    - tensor: A PyTorch tensor of the image, expected shape: (channels, height, width).
    - one_channel: Set to True if the tensor contains a single channel (grayscale image).
    """
    # Clone the tensor to not do changes on the original
    tensor = tensor.clone().detach()
    # Convert to numpy array
    numpy_image = tensor.numpy()

    if one_channel:
        numpy_image = numpy_image.squeeze()  # Remove single-dimensional entries from the shape
        plt.imshow(numpy_image, cmap="gray")
    else:
        # Transpose the tensor to (height, width, channels)
        numpy_image = np.transpose(numpy_image, (1, 2, 0))
        plt.imshow(numpy_image)
    plt.show()

for i in range(16):
    image_tensor = train_features[i]
    show_image(image_tensor, one_channel=False)

## Data Processing for Baseline (Random Forest): CSV iles

In [ ]:
data_30sec = pd.read_csv('/content/drive/My Drive/APS360 Project/Data/features_30_sec.csv')
data_30sec.head()

In [ ]:
data = pd.read_csv('/content/drive/My Drive/APS360 Project/Data/features_3_sec.csv')
data.head()

Since generally the more data we have the better our results will be, we decide to proceed with features_3_sec.csv dataset (9990 rows of data) to train and evaluate our models.

In [ ]:
# count the number of songs for each genre
genre_counts = data['label'].value_counts()

# print the counts for each class
print(genre_counts)

In [ ]:
# plotting the genre distribution histogram
plt.figure(figsize=(10, 6))
plt.bar(genre_counts.index, genre_counts.values)
plt.xlabel('Class (Genre)')
plt.ylabel('Number of Songs')
plt.title('Distribution of Songs Across Different Genres')
plt.xticks(rotation=45)
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

As the number of songs per genre are all equal or extremely close to 1000 (classes are balanced), we can directly use the features_3_sec.csv for our model training and testing.

In [ ]:
data.head()

In [ ]:
data.columns

In [ ]:
from sklearn.preprocessing import MinMaxScaler

df = data.iloc[0:, 2:] # get rid of filename and length columns as they are not useful features for prediction
target = df.label.values # save all the genres into target

features = df.drop('label', axis = 1) # save all the features into label
features_columns = features.columns

# scaling the features using MinMaxScaler
# normalize so everything is on the same scale
scale = MinMaxScaler()
scaled_data = scale.fit_transform(features)
features = pd.DataFrame(scaled_data, columns = features_columns).values

In [ ]:
features

In [ ]:
features.shape

In [ ]:
features_columns

In [ ]:
target

In [ ]:
target.shape

In [ ]:
from sklearn.model_selection import StratifiedShuffleSplit

X = features
y = target

# stratified splitting with a ratio of 60% training, 20% validation, and 20% testing
splitter = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=42)

# splitting data into training and remaining data (which will be further split into validation and testing)
for train_index, remaining_index in splitter.split(X, y):
    X_train, X_remaining = X[train_index], X[remaining_index]
    y_train, y_remaining = y[train_index], y[remaining_index]

# further splitting remaining data into validation and testing
splitter = StratifiedShuffleSplit(n_splits=1, test_size=0.5, random_state=42)

for val_index, test_index in splitter.split(X_remaining, y_remaining):
    X_val, X_test = X_remaining[val_index], X_remaining[test_index]
    y_val, y_test = y_remaining[val_index], y_remaining[test_index]

print("Training set shape:", X_train.shape, y_train.shape)
print("Validation set shape:", X_val.shape, y_val.shape)
print("Testing set shape:", X_test.shape, y_test.shape)


In [ ]:
# (training set) make sure the music genres are still distributed evenly

# count the number of occurrences of each class
classes, counts = np.unique(y_train, return_counts=True)

for c, count in zip(classes, counts):
    print(f"Class {c}: {count} instances")

In [ ]:
# (validation set) make sure the music genres are still distributed evenly

# count the number of occurrences of each class
classes, counts = np.unique(y_val, return_counts=True)

for c, count in zip(classes, counts):
    print(f"Class {c}: {count} instances")

In [ ]:
# (testing set) make sure the music genres are still distributed evenly

# count the number of occurrences of each class
classes, counts = np.unique(y_test, return_counts=True)

for c, count in zip(classes, counts):
    print(f"Class {c}: {count} instances")

# Baseline Model: Random Forest

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, precision_recall_fscore_support
import matplotlib.pyplot as plt
import seaborn as sns

# train the Random Forest Classifier
rf_classifier = RandomForestClassifier(n_estimators=1000, max_depth=10, random_state=42)
rf_classifier.fit(X_train, y_train)

# predict on validation set - assuming the test set will not be available until the very end of the project
y_val_pred = rf_classifier.predict(X_val)

# calculate accuracy
accuracy = accuracy_score(y_val, y_val_pred)
print("Validation Accuracy:", accuracy)

# calculate precision, recall, and f1-score
precision, recall, f1_score, _ = precision_recall_fscore_support(y_val, y_val_pred, average='weighted')
print("Precision:", precision)
print("Recall:", recall)
print("F1-score:", f1_score)

# create multiclass confusion matrix
cm = confusion_matrix(y_val, y_val_pred)

# plot confusion matrix
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=np.unique(y), yticklabels=np.unique(y))
plt.xlabel('Predicted labels')
plt.ylabel('True labels')
plt.title('Confusion Matrix')
plt.show()


# Primary Model: Convolutional Neural Network (CNN)

In [ ]:
import numpy as np
import time
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision
from torch.utils.data.sampler import SubsetRandomSampler
import torchvision.transforms as transforms
import torchvision.datasets as datasets
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
from PIL import Image

In [ ]:
###############################################################################
def evaluate(net, loader, criterion):
    """ Evaluate the network on the validation set.

     Args:
         net: PyTorch neural network object
         loader: PyTorch data loader for the validation set
         criterion: The loss function
     Returns:
         err: A scalar for the avg classification error over the validation set
         loss: A scalar for the average loss function over the validation set
     """
    total_loss = 0.0
    total_err = 0.0
    total_epoch = 0
    for i, data in enumerate(loader, 0):
        inputs, labels = data

        outputs = net(inputs)
        loss = criterion(outputs, labels)
        _, predicted = torch.max(outputs, 1)  # Get the indices of the max log-probability
        total_err += (predicted != labels).sum().item()
        total_loss += loss.item()
        total_epoch += len(labels)
    err = float(total_err) / total_epoch
    loss = float(total_loss) / (i + 1)
    return err, loss

###############################################################################
# Training
def get_model_name(name, batch_size, learning_rate, epoch):
    """ Generate a name for the model consisting of all the hyperparameter values

    Args:
        config: Configuration object containing the hyperparameters
    Returns:
        path: A string with the hyperparameter name and value concatenated
    """
    path = "model_{0}_bs{1}_lr{2}_epoch{3}".format(name,
                                                   batch_size,
                                                   learning_rate,
                                                   epoch)
    return path

def get_accuracy(model, data_loader):
    correct = 0
    total = 0

    for imgs, labels in data_loader:

        output = model(imgs)

        # Select index with maximum prediction score
        pred = output.max(1, keepdim=True)[1]
        correct += pred.eq(labels.view_as(pred)).sum().item()
        total += labels.size(0)

    return correct / total

###############################################################################
# Training Curve
def plot_training_curve_path(path):
    """ Plots the training curve for a model run, given the csv files
    containing the train/validation error/loss.

    Args:
        path: The base path of the csv files produced during training
    """
    import matplotlib.pyplot as plt
    train_err = np.loadtxt("{}_train_err.csv".format(path))
    val_err = np.loadtxt("{}_val_err.csv".format(path))
    train_loss = np.loadtxt("{}_train_loss.csv".format(path))
    val_loss = np.loadtxt("{}_val_loss.csv".format(path))
    plt.title("Train vs Validation Error")
    n = len(train_err) # number of epochs
    plt.plot(range(1,n+1), train_err, label="Train")
    plt.plot(range(1,n+1), val_err, label="Validation")
    plt.xlabel("Epoch")
    plt.ylabel("Error")
    plt.legend(loc='best')
    plt.show()
    plt.title("Train vs Validation Loss")
    plt.plot(range(1,n+1), train_loss, label="Train")
    plt.plot(range(1,n+1), val_loss, label="Validation")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.legend(loc='best')
    plt.show()


def plot_training_curve(train, val, type):
    """ Plots the training curve for a model run, given the csv files
    containing the train/validation error/loss.

    Args:
        path: The base path of the csv files produced during training
    """
    import matplotlib.pyplot as plt

    plt.title(f"Train vs Validation {type}")
    n = len(train) # number of epochs

    plt.plot(range(1,n+1), train, label="Train")
    plt.plot(range(1,n+1), val, label="Validation")
    plt.xlabel("Epoch")
    plt.ylabel(f"{type}")
    plt.legend(loc='best')
    plt.show()

###############################################################################
import numpy as np
import seaborn as sns
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score

classes = ['blues', 'classical', 'country', 'disco', 'hiphop', 'jazz', 'metal', 'pop', 'reggae', 'rock']

def evaluate_and_plot(model, test_loader, classes):
    all_preds = []
    all_labels = []
    with torch.no_grad():  # No need to track the gradients
        for images, labels in test_loader:
            outputs = model(images)
            _, predicted = torch.max(outputs, 1)
            all_preds.extend(predicted.numpy())
            all_labels.extend(labels.numpy())

    # Compute the confusion matrix
    cm = confusion_matrix(all_labels, all_preds)
    # Compute metrics
    accuracy = accuracy_score(all_labels, all_preds)
    precision = precision_score(all_labels, all_preds, average='macro')
    recall = recall_score(all_labels, all_preds, average='macro')
    f1 = f1_score(all_labels, all_preds, average='macro')

    # Print the metrics
    print(f'Precision: {precision:.4f}')
    print(f'Recall: {recall:.4f}')
    print(f'F1-score: {f1:.4f}')

    # Plotting the confusion matrix
    plt.figure(figsize=(10,8))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=classes, yticklabels=classes)
    plt.title('Confusion Matrix')
    plt.ylabel('True labels')
    plt.xlabel('Predicted labels')
    plt.show()

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy

class CNN1(nn.Module):
    def __init__(self, conv_layers, conv_out_channels, fc_layers, num_classes=10, input_size=64, name='CNN1'):
        super(CNN1, self).__init__()
        self.name = name

        # Initialize convolutional layers container
        self.conv_layers_list = nn.ModuleList()

        # Dynamically create convolutional layers
        in_channels = 3  # RGB images
        for i in range(conv_layers):
            # Double the out_channels for every layer after the first
            out_channels = conv_out_channels * (2 ** i)
            conv_layer = nn.Conv2d(in_channels=in_channels, out_channels=out_channels, kernel_size=3, stride=1, padding=1)
            self.conv_layers_list.append(conv_layer)
            in_channels = out_channels  # Set the in_channels for the next layer

        # Max pooling layer (common for all conv layers)
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2, padding=0)

        # Dropout
        self.dropout = nn.Dropout(0.3)  # Dropout rate of 30%

        # Dynamically compute the flattened size after convolutions and pooling
        self._num_flat_features = self._get_conv_output((3, input_size, input_size))  # Corrected to pass a shape tuple

        # Fully connected layers
        self.fc_layers = nn.ModuleList()
        for i, output_features in enumerate(fc_layers):
            input_features = self._num_flat_features if i == 0 else fc_layers[i - 1]
            self.fc_layers.append(nn.Linear(in_features=input_features, out_features=output_features))

        # Output layer
        self.output_layer = nn.Linear(in_features=fc_layers[-1], out_features=num_classes)

    def _get_conv_output(self, shape):
        # Create a dummy input with the input size
        batch_size = 1
        input = torch.autograd.Variable(torch.rand(batch_size, *shape))
        # Apply convolutional layers and pooling layers
        for conv in self.conv_layers_list:
            input = conv(input)
            input = self.pool(input)
        # Flatten the output
        return int(numpy.prod(input.size()[1:]))

    def forward(self, x):
        # Apply convolutional layers and pooling layers with dropout
        for conv in self.conv_layers_list:
            x = self.dropout(self.pool(F.relu(conv(x))))

        # Flatten the output for the fully connected layer
        x = x.view(-1, self._num_flat_features)

        # Apply fully connected layers with ReLU activation and dropout
        for fc in self.fc_layers:
            x = F.relu(fc(x))

        # Output layer
        x = self.output_layer(x)

        return x


In [ ]:
model = CNN1(conv_layers=6, conv_out_channels=6, fc_layers=[1024, 512, 128, 32], num_classes=10, input_size=64, name='CustomCNN')

In [ ]:
def count_parameters(net):
    total_params = 0
    for param in net.parameters():
        # Number of parameters in this layer
        layer = 1
        for dim in param.shape:
            layer *= dim
        total_params += layer
    return total_params

print(f"Total parameters in network: {count_parameters(model)}")

In [ ]:
import time

def train(model, batch_size, num_epochs, learning_rate):

    ########################################################################
    torch.manual_seed(42)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.SGD(model.parameters(), learning_rate, momentum=0.9)
    ########################################################################
    # Use the function to get data loaders
    folder_path = '/content/drive/MyDrive/APS360 Project/Data/images_original'
    train_loader, val_loader, test_loader = get_loaders(folder_path, batch_size)
    ########################################################################
    # Set up some numpy arrays to store the training/test loss/erruracy
    train_err = np.zeros(num_epochs)
    train_loss = np.zeros(num_epochs)
    train_acc = np.zeros(num_epochs)
    val_err = np.zeros(num_epochs)
    val_loss = np.zeros(num_epochs)
    val_acc = np.zeros(num_epochs)
    ########################################################################
    # param to print time for approx 1 epoch
    tt = 0

    for epoch in range(num_epochs):
        tt = tt + 1
        start_time = time.time()
        #print(f"Epoch {epoch+1}/{num_epochs}")
        total_train_loss = 0.0
        total_train_err = 0.0
        total_train_correct = 0.0
        total_itr = 0
        total_train_data = 0
        for i, data in enumerate(train_loader, 0):
            # Get the inputs
            imgs, labels = data

            out = model(imgs)             # forward pass

            loss = criterion(out, labels) # compute the total loss
            loss.backward()               # backward pass (compute parameter updates)
            optimizer.step()              # make the updates for each parameter
            optimizer.zero_grad()         # a clean up step for PyTorch

            # Calculate the statistics
            _, predicted = torch.max(out, 1)  # Get the indices of the max log-probability
            total_train_err += (predicted != labels).sum().item()
            total_train_correct += (predicted == labels).sum().item()
            total_train_loss += loss.item()
            total_train_data += labels.size(0)
            total_itr += len(labels)

        # Calculate the average statistics
        train_err[epoch] = float(total_train_err) / total_itr
        train_loss[epoch] = float(total_train_loss) / (i+1)
        val_err[epoch], val_loss[epoch] = evaluate(model, val_loader, criterion)
        train_acc[epoch] = get_accuracy(model, train_loader)
        val_acc[epoch] = get_accuracy(model, val_loader)

        if tt < 2:
          end_time = time.time()
          diff = end_time - start_time
          print(f"Approximate time for 1 epoch: {diff}")

  ############################################################################
    classes = ['blues', 'classical', 'country', 'disco', 'hiphop', 'jazz', 'metal', 'pop', 'reggae', 'rock']
    evaluate_and_plot(model, val_loader, classes)

    # Save the current model (checkpoint) to a file
    model_path = get_model_name(model.name, batch_size, learning_rate, num_epochs)
    torch.save(model.state_dict(), model_path)
    # Write the train/test loss/err into CSV file for plotting later
    epochs = np.arange(1, num_epochs + 1)
    np.savetxt("{}_train_err.csv".format(model_path), train_err)
    np.savetxt("{}_train_loss.csv".format(model_path), train_loss)
    np.savetxt("{}_val_err.csv".format(model_path), val_err)
    np.savetxt("{}_val_loss.csv".format(model_path), val_loss)
    np.savetxt("{}_train_acc.csv".format(model_path), train_acc)
    np.savetxt("{}_val_acc.csv".format(model_path), val_acc)
    ########################################################################

    return train_acc, val_acc, test_loader

# Hyperparameter Tunning for CNN

In [ ]:
# default: lr=1e-2, bs=32, out_channel=6, fc=[1024, 512, 128, 64]

In [ ]:
# default
model_set1 = CNN1(conv_layers=6, conv_out_channels=6, fc_layers=[1024, 512, 128, 64], num_classes=10, input_size=64, name='model_set1')
train_acc_set1, val_acc_set1, test_loader_set1 = train(model_set1, batch_size=32, num_epochs=60, learning_rate=1e-2)

model_path = get_model_name('model_set1', batch_size=32, learning_rate=1e-2, epoch=60)
print(model_path)
plot_training_curve_path(model_path)

In [ ]:
# default
model_set1 = CNN1(conv_layers=6, conv_out_channels=6, fc_layers=[1024, 512, 128, 64], num_classes=10, input_size=64, name='model_set1')
train_acc_set1, val_acc_set1, test_loader_set1 = train(model_set1, batch_size=32, num_epochs=100, learning_rate=1e-2)

model_path = get_model_name('model_set1', batch_size=32, learning_rate=1e-2, epoch=100)
print(model_path)
plot_training_curve_path(model_path)

In [ ]:
# default
model_set1 = CNN1(conv_layers=6, conv_out_channels=6, fc_layers=[1024, 512, 128, 64], num_classes=10, input_size=64, name='model_set1')
train_acc_set1, val_acc_set1, test_loader_set1 = train(model_set1, batch_size=32, num_epochs=200, learning_rate=1e-2)

model_path = get_model_name('model_set1', batch_size=32, learning_rate=1e-2, epoch=200)
print(model_path)
plot_training_curve_path(model_path)

In [ ]:
# model_set2
# conv_out_channels=12
model_set2 = CNN1(conv_layers=6, conv_out_channels=12, fc_layers=[1024, 512, 128, 64], num_classes=10, input_size=64, name='model_set2')
train_acc_set2, val_acc_set2, test_loader_set2 = train(model_set2, batch_size=32, num_epochs=100, learning_rate=1e-2)

model_path = get_model_name('model_set2', batch_size=32, learning_rate=1e-2, epoch=100)
print(model_path)
plot_training_curve_path(model_path)

In [ ]:
print(train_acc_set2[99], val_acc_set2[99])

In [ ]:
# model_set3
# fc_layers=[512, 128, 64]
model_set3 = CNN1(conv_layers=6, conv_out_channels=6, fc_layers=[512, 128, 64], num_classes=10, input_size=64, name='model_set3')
train_acc_set3, val_acc_set3, test_loader_set3 = train(model_set3, batch_size=32, num_epochs=100, learning_rate=1e-2)

model_path = get_model_name('model_set3', batch_size=32, learning_rate=1e-2, epoch=100)
print(model_path)
plot_training_curve_path(model_path)

In [ ]:
# model_set4
# lr=1e-3
model_set4 = CNN1(conv_layers=6, conv_out_channels=6, fc_layers=[1024, 512, 128, 64], num_classes=10, input_size=64, name='model_set4')
train_acc_set4, val_acc_set4, test_loader_set4 = train(model_set4, batch_size=32, num_epochs=100, learning_rate=1e-3)

model_path = get_model_name('model_set4', batch_size=32, learning_rate=1e-3, epoch=100)
print(model_path)
plot_training_curve_path(model_path)

In [ ]:
# model_set5
# bs=64
model_set5 = CNN1(conv_layers=6, conv_out_channels=6, fc_layers=[1024, 512, 128, 64], num_classes=10, input_size=64, name='model_set5')
train_acc_set5, val_acc_set5, test_loader_set5 = train(model_set5, batch_size=64, num_epochs=100, learning_rate=1e-2)

model_path = get_model_name('model_set5', batch_size=64, learning_rate=1e-2, epoch=100)
print(model_path)
plot_training_curve_path(model_path)

In [ ]:
print(train_acc_set5[99], val_acc_set5[99])

In [ ]:
# model_set6
# conv_out_channels=12, fc_layers=[512, 128, 64]
model_set6 = CNN1(conv_layers=6, conv_out_channels=12, fc_layers=[512, 128, 64], num_classes=10, input_size=64, name='model_set6')
train_acc_set6, val_acc_set6, test_loader_set6 = train(model_set6, batch_size=32, num_epochs=100, learning_rate=1e-2)

model_path = get_model_name('model_set6', batch_size=32, learning_rate=1e-2, epoch=100)
print(model_path)
plot_training_curve_path(model_path)

In [ ]:
print(train_acc_set6[99], val_acc_set6[99])

In [ ]:
# model_set7
# conv_out_channels=12, lr=1e-3
model_set7 = CNN1(conv_layers=6, conv_out_channels=12, fc_layers=[1024, 512, 128, 64], num_classes=10, input_size=64, name='model_set7')
train_acc_set7, val_acc_set7, test_loader_set7 = train(model_set7, batch_size=32, num_epochs=100, learning_rate=1e-3)

model_path = get_model_name('model_set7', batch_size=32, learning_rate=1e-3, epoch=100)
print(model_path)
plot_training_curve_path(model_path)

In [ ]:
print(train_acc_set7[99], val_acc_set7[99])

In [ ]:
# model_set8
# conv_out_channels=12, bs=64
model_set8 = CNN1(conv_layers=6, conv_out_channels=12, fc_layers=[1024, 512, 128, 64], num_classes=10, input_size=64, name='model_set8')
train_acc_set8, val_acc_set8, test_loader_set8 = train(model_set8, batch_size=64, num_epochs=100, learning_rate=1e-2)

model_path = get_model_name('model_set8', batch_size=64, learning_rate=1e-2, epoch=100)
print(model_path)
plot_training_curve_path(model_path)

In [ ]:
print(train_acc_set8[99], val_acc_set8[99])

In [ ]:
# model_set9
# fc_layers=[512, 128, 64], lr=1e-3
model_set9 = CNN1(conv_layers=6, conv_out_channels=6, fc_layers=[512, 128, 64], num_classes=10, input_size=64, name='model_set9')
train_acc_set9, val_acc_set9, test_loader_set9 = train(model_set9, batch_size=32, num_epochs=100, learning_rate=1e-3)

model_path = get_model_name('model_set9', batch_size=32, learning_rate=1e-3, epoch=100)
print(model_path)
plot_training_curve_path(model_path)

In [ ]:
print(train_acc_set9[99], val_acc_set9[99])

In [ ]:
# model_set10
# fc_layers=[512, 128, 64], bs=64
model_set10 = CNN1(conv_layers=6, conv_out_channels=6, fc_layers=[512, 128, 64], num_classes=10, input_size=64, name='model_set10')
train_acc_set10, val_acc_set10, test_loader_set10 = train(model_set10, batch_size=64, num_epochs=100, learning_rate=1e-2)

model_path = get_model_name('model_set10', batch_size=64, learning_rate=1e-2, epoch=100)
print(model_path)
plot_training_curve_path(model_path)

In [ ]:
print(train_acc_set10[99], val_acc_set10[99])

In [ ]:
# model_set11
# lr=1e-3, bs=64
model_set11 = CNN1(conv_layers=6, conv_out_channels=6, fc_layers=[1024, 512, 128, 64], num_classes=10, input_size=64, name='model_set11')
train_acc_set11, val_acc_set11, test_loader_set11 = train(model_set11, batch_size=64, num_epochs=100, learning_rate=1e-3)

model_path = get_model_name('model_set11', batch_size=64, learning_rate=1e-3, epoch=100)
print(model_path)
plot_training_curve_path(model_path)

In [ ]:
print(train_acc_set11[99], val_acc_set11[99])

In [ ]:
# model_set12
# conv_out_channels=12, fc_layers=[512, 128, 64], lr=1e-3
model_set12 = CNN1(conv_layers=6, conv_out_channels=12, fc_layers=[512, 128, 64], num_classes=10, input_size=64, name='model_set12')
train_acc_set12, val_acc_set12, test_loader_set12 = train(model_set12, batch_size=32, num_epochs=100, learning_rate=1e-3)

model_path = get_model_name('model_set12', batch_size=32, learning_rate=1e-3, epoch=100)
print(model_path)
plot_training_curve_path(model_path)

In [ ]:
print(train_acc_set12[99], val_acc_set12[99])

In [ ]:
# model_set13
# conv_out_channels=12, fc_layers=[512, 128, 64], bs=64
model_set13 = CNN1(conv_layers=6, conv_out_channels=12, fc_layers=[512, 128, 64], num_classes=10, input_size=64, name='model_set13')
train_acc_set13, val_acc_set13, test_loader_set13 = train(model_set13, batch_size=64, num_epochs=100, learning_rate=1e-2)

model_path = get_model_name('model_set13', batch_size=64, learning_rate=1e-2, epoch=100)
print(model_path)
plot_training_curve_path(model_path)

In [ ]:
print(train_acc_set13[99], val_acc_set13[99])

In [ ]:
# model_set14
# fc_layers=[512, 128, 64], lr=1e-3, bs=64
model_set14 = CNN1(conv_layers=6, conv_out_channels=6, fc_layers=[512, 128, 64], num_classes=10, input_size=64, name='model_set14')
train_acc_set14, val_acc_set14, test_loader_set14 = train(model_set14, batch_size=64, num_epochs=100, learning_rate=1e-3)

model_path = get_model_name('model_set14', batch_size=64, learning_rate=1e-3, epoch=100)
print(model_path)
plot_training_curve_path(model_path)

In [ ]:
print(train_acc_set14[99], val_acc_set14[99])

In [ ]:
# model_set15
# conv_out_channels=12, fc_layers=[512, 128, 64], lr=1e-3, bs=64
model_set15 = CNN1(conv_layers=6, conv_out_channels=12, fc_layers=[512, 128, 64], num_classes=10, input_size=64, name='model_set15')
train_acc_set15, val_acc_set15, test_loader_set15 = train(model_set15, batch_size=64, num_epochs=100, learning_rate=1e-3)

model_path = get_model_name('model_set15', batch_size=64, learning_rate=1e-3, epoch=100)
print(model_path)
plot_training_curve_path(model_path)

In [ ]:
print(train_acc_set15[99], val_acc_set15[99])

# Evaluation for CNN: test_loader

In [ ]:
err, loss = evaluate(model_set13, test_loader_set13, nn.CrossEntropyLoss())
acc = get_accuracy(model_set13, test_loader_set13)
print(err, loss, acc)

evaluate_and_plot(model_set13, test_loader_set13, classes)

In [ ]:
print(f"train acc: {train_acc_set13[99]}") # from hyperparameter tuning section
print(f"val acc: {val_acc_set13[99]}") # from hyperparameter tuning section
print(f"test acc: {acc}")

# Evaluation for CNN: Real world data

In [ ]:
import os
import matplotlib.pyplot as plt
import librosa
import librosa.display
import numpy as np

# Define the genres
class_names = ['blues', 'classical', 'country', 'disco', 'hiphop', 'jazz', 'metal', 'pop', 'reggae', 'rock']

# Define the directories
source_dir = '/content/drive/MyDrive/APS360 Project/Data_evaluate/'
target_dir = '/content/drive/MyDrive/APS360 Project/spectrogram_evaluate/'

# Ensure the target directory exists
if not os.path.exists(target_dir):
    os.makedirs(target_dir)

def create_save_melspectrogram(data, sr, start_sec, end_sec, genre, file_index, target_dir):
    # Slice the audio data for the specified segment
    start_sample = int(start_sec * sr)
    end_sample = int(end_sec * sr)
    data_segment = data[start_sample:end_sample]

    # Generate a mel-scaled spectrogram
    S = librosa.feature.melspectrogram(y=data_segment, sr=sr, n_mels=128, fmax=8000)

    # Convert the mel spectrogram to log scale (for better visual representation)
    log_S = librosa.power_to_db(S, ref=np.max)

    # Create a figure
    plt.figure(figsize=(10, 4))

    # Display the mel spectrogram
    librosa.display.specshow(log_S, sr=sr, x_axis='time', y_axis='mel', fmax=8000)

    # Remove the axis
    plt.axis('off')

    # Check and create genre subdirectory if necessary
    genre_target_dir = os.path.join(target_dir, genre)
    if not os.path.exists(genre_target_dir):
        os.makedirs(genre_target_dir)

    # Format the file index with leading zeros
    formatted_index = f"{file_index:05d}"

    # Save the plot without frame
    output_filename = f"{genre}{formatted_index}.png"
    output_filepath = os.path.join(genre_target_dir, output_filename)
    plt.savefig(output_filepath, bbox_inches='tight', pad_inches=0)  # Save without padding and frame

    # Close the plot figure to free up memory
    plt.close()

# Initialize a counter for the file index outside of the file loop
index_counter = {}

# Iterate over each genre
for genre in class_names:
    print(f"Processing genre: {genre}")
    genre_source_dir = os.path.join(source_dir, genre)
    genre_target_dir = os.path.join(target_dir, genre)

    # Ensure the target genre directory exists
    if not os.path.exists(genre_target_dir):
        os.makedirs(genre_target_dir)

    # Initialize the counter for each genre
    index_counter[genre] = 0

    # Iterate over each file in the genre directory
    for filename in os.listdir(genre_source_dir):
        if filename.endswith('.mp3'):
            file_path = os.path.join(genre_source_dir, filename)

            # Load the audio file
            data, sr = librosa.load(file_path, sr=None)

            # Create and save spectrograms for the specified segments
            # Use the genre-specific index for filename
            create_save_melspectrogram(data, sr, 0, 30, genre, index_counter[genre], target_dir)
            # Increment the index for the next file
            index_counter[genre] += 1
            create_save_melspectrogram(data, sr, 60, 90, genre, index_counter[genre], target_dir)

            # Increment the index for the next file
            index_counter[genre] += 1

print("Processing complete.")

In [ ]:
folder_path_evaluate = '/content/drive/MyDrive/APS360 Project/spectrogram_evaluate'

transform = transforms.Compose([
    transforms.Lambda(lambda x: x.convert('RGB')),  # Convert image to RGB
    transforms.Resize((64, 64)),  # Resize each slice to 64x64 - smaller than before as we have more samples to feed into CNN
    transforms.ToTensor()
])

# Initialize the custom dataset
dataset = ChoppedDataset(folder_path=folder_path_evaluate, transform=transform)

# Create data loaders for each set
evaluate_loader = DataLoader(dataset, batch_size=64, shuffle=True)

# Evaluate the model
err, loss = evaluate(model_set13, evaluate_loader, nn.CrossEntropyLoss())
acc = get_accuracy(model_set13, evaluate_loader)

print(f"Error: {err}")
print(f"Loss: {loss}")
print(f"Accuracy: {acc}")

# Qualitative Analysis on Test Set
- To provide a comprehensive understanding of our model's performance, we have elected to present outputs that highlight both its strengths and limitations.
- We showcased the model's performance by comparing predicted and actual labels against actual spectrograms for 'country' (where the model struggles) and 'pop' (where it excels), thereby illuminating its capabilities and shortcomings.

In [ ]:
from torchvision.utils import make_grid

# Display images
def imshow(img):
    npimg = img.numpy()
    plt.imshow(np.transpose(npimg, (1, 2, 0)))
    plt.show()

# Function to find and display true and false predictions for specified classes
def find_predictions(test_loader, model, class_names, class_of_interest, num_examples=2):
    model.eval()  # Set the model to evaluation mode
    true_examples = []
    false_examples = []

    with torch.no_grad():  # No need to track gradients
        for images, labels in test_loader:
            outputs = model(images)
            _, predicted = torch.max(outputs, 1)

            for image, label, prediction in zip(images, labels, predicted):
                if class_names[label] == class_of_interest:
                    if label == prediction and len(true_examples) < num_examples:
                        true_examples.append((image, prediction))
                    elif label != prediction and len(false_examples) < num_examples:
                        false_examples.append((image, prediction))

                # Check if we have found enough examples
                if len(true_examples) >= num_examples and len(false_examples) >= num_examples:
                    break
            else:
                # Continue outer loop if inner loop wasn't broken
                continue
            # Inner loop was broken, break the outer.
            break

    # Display true examples
    print(f"True predictions for class '{class_of_interest}':")
    for image, prediction in true_examples:
        print(f"Predicted: '{class_names[prediction]}' - Correct!")
        if image.dim() == 3:  # If image is a 3D tensor (C, H, W)
            imshow(make_grid(image.unsqueeze(0)))  # Add a batch dimension and display
        else:
            imshow(make_grid(image))  # Display image

    # Display false examples
    print(f"False predictions for class '{class_of_interest}':")
    for image, prediction in false_examples:
        print(f"Predicted: '{class_names[prediction]}' - Incorrect!")
        if image.dim() == 3:  # If image is a 3D tensor (C, H, W)
            imshow(make_grid(image.unsqueeze(0)))  # Add a batch dimension and display
        else:
            imshow(make_grid(image))  # Display image

class_names = ['blues', 'classical', 'country', 'disco', 'hiphop', 'jazz', 'metal', 'pop', 'reggae', 'rock']

find_predictions(test_loader, model_set13, class_names, 'pop')
find_predictions(test_loader, model_set13, class_names, 'country')